Load Packages 

In [ ]:
%pip install openpyxl

In [ ]:
import re
import os
from pathlib import Path
from dataclasses import dataclass
from typing import Dict, List, Tuple, Optional

import pandas as pd

Create Dictionaries

In [ ]:
CORE_AI_TERMS = [

    # ---- Generic AI ----
    "artificial intelligence",
    "ai",  # standalone only (handled by word-boundary regex)
    "generative artificial intelligence",
    "generative ai",
    "genai",
    "ai-driven",
    "ai powered",
    "ai-enabled",
    "ai-based",
    "intelligent automation",
    "autonomous ai system",
    "machine intelligence",
    "ai agent",

    # ---- Machine Learning ----
    "machine learning",
    "ml",  # standalone only
    "deep learning",
    "neural network",
    "neural networks",
    "artificial neural network",
    "ann",
    "convolutional neural network",
    "cnn",
    "recurrent neural network",
    "rnn",
    "supervised learning",
    "unsupervised learning",
    "semi-supervised learning",
    "reinforcement learning",
    "transfer learning",
    "adversarial machine learning",
    "generative adversarial network",
    "gan",
    "expert system",
    "adaptive algorithm",

    # ---- Large Language Models ----
    "large language model",
    "large language models",
    "llm",
    "llms",
    "foundation model",
    "foundation models",
    "transformer model",
    "generative model",
    "text generation",
    "language model",
    "conversational ai",

    # ---- Named Technologies ----
    "chatgpt",
    "gpt",
    "gpt-3",
    "gpt-4",
    "openai",
    "copilot",
    "bard",
    "gemini",
    "claude",

    # ---- Computer Vision ----
    "computer vision",
    "image recognition",
    "object detection",
    "facial recognition",
    "optical character recognition",
    "ocr",
    "emotion recognition system",

    # ---- NLP ----
    "natural language processing",
    "nlp",  # standalone only
    "natural language understanding",
    "nlu",
    "speech recognition",
    "chatbot"
]


AI_ADJACENT_TERMS = [

    # ---- Automation & Algorithms ----
    "automation",
    "automated",
    "algorithm",
    "algorithms",
    "predictive analytics",
    "advanced analytics",

    # ---- Analytics ----
    "analytics",
    "data analytics",
    "data science",
    "decision intelligence",
    "intelligent systems",
    "recommendation system",
    "predictive analysis",
    "profiling",
    "adaptive learning",
    "automated decision-making",
    "bot",

    # ---- Digital Transformation ----
    "digital transformation",
    "digitalization",
    "digitization",
    "smart systems",
    "industry 4.0",
    "intelligent platform",

    # ---- Robotics & Process Tech ----
    "robotics",
    "robotic process automation",
    "rpa",
    "autonomous systems",

    # ---- Data & Infrastructure ----
    "big data",
    "data-driven",
    "advanced computing",
    "cloud-based intelligence",
    "high-performance computing",
    "ai infrastructure",
    "structured data",
    "unstructured data",
    "input data",

    # ---- Enterprise AI Integration ----
    "ai integration",
    "ai platform",
    "ai capabilities",
    "ai deployment",
    "ai applications",
    "ai strategy",
    "ai solutions",
    "ai initiative",
    "ai roadmap"
]

Transcript  parsing: splitting Presentation and Q&A

In [ ]:
PRESENTATION_HEADER = r"^Presentation\s*$"
QA_HEADER = r"^Questions and Answers\s*$"

SECTION_DIVIDER_LINE = r"^-{20,}\s*$"  # many transcripts use long dashed lines


def split_presentation_and_qa(text: str) -> Tuple[str, str]:
    """
    Returns (presentation_text, qa_text).
    If a section isn't found, returns "" for that section.
    """
    lines = text.splitlines()

    # Find the first occurrence of "Presentation" and "Questions and Answers" as standalone lines
    pres_idx = None
    qa_idx = None

    for i, line in enumerate(lines):
        if pres_idx is None and re.match(PRESENTATION_HEADER, line.strip(), flags=re.IGNORECASE):
            pres_idx = i
        if qa_idx is None and re.match(QA_HEADER, line.strip(), flags=re.IGNORECASE):
            qa_idx = i

    if pres_idx is None and qa_idx is None:
        return "", ""

    # Presentation section usually starts after the "Presentation" header and its divider lines
    if pres_idx is not None:
        pres_start = pres_idx + 1
        # Skip divider lines right after the header
        while pres_start < len(lines) and (
            re.match(SECTION_DIVIDER_LINE, lines[pres_start].strip())
            or lines[pres_start].strip() == ""
        ):
            pres_start += 1
    else:
        pres_start = None

    # Q&A section usually starts after the "Questions and Answers" header and its divider lines
    if qa_idx is not None:
        qa_start = qa_idx + 1
        while qa_start < len(lines) and (
            re.match(SECTION_DIVIDER_LINE, lines[qa_start].strip())
            or lines[qa_start].strip() == ""
        ):
            qa_start += 1
    else:
        qa_start = None

    # Determine ranges
    if pres_start is not None and qa_start is not None:
        presentation = "\n".join(lines[pres_start:qa_idx]).strip()
        qa = "\n".join(lines[qa_start:]).strip()
        return presentation, qa

    if pres_start is not None and qa_start is None:
        # No Q&A found; treat everything after presentation header as presentation
        presentation = "\n".join(lines[pres_start:]).strip()
        return presentation, ""

    if pres_start is None and qa_start is not None:
        # No presentation found; treat everything after Q&A header as Q&A
        qa = "\n".join(lines[qa_start:]).strip()
        return "", qa

    return "", ""


Tokenization + dicitonary hit counts

In [ ]:
WORD_RE = re.compile(r"[A-Za-z0-9]+(?:'[A-Za-z0-9]+)?")  # simple, robust

def normalize_for_matching(s: str) -> str:
    # Lowercase and collapse whitespace
    return re.sub(r"\s+", " ", s.lower()).strip()

def word_count(text: str) -> int:
    return len(WORD_RE.findall(text))

def build_term_patterns(terms: List[str]) -> List[Tuple[str, re.Pattern]]:
    """
    Build regex patterns that count whole-word matches.
    - For single tokens like "ai" or "ml", we use word boundaries.
    - For phrases, we allow flexible whitespace.
    """
    patterns = []
    for t in terms:
        t_norm = normalize_for_matching(t)
        if " " in t_norm:
            # Phrase: replace spaces with \s+ and enforce word boundaries at ends
            phrase = re.escape(t_norm).replace(r"\ ", r"\s+")
            pat = re.compile(rf"\b{phrase}\b", flags=re.IGNORECASE)
        else:
            # Single token
            pat = re.compile(rf"\b{re.escape(t_norm)}\b", flags=re.IGNORECASE)
        patterns.append((t, pat))
    return patterns

CORE_PATTERNS = build_term_patterns(CORE_AI_TERMS)
ADJ_PATTERNS = build_term_patterns(AI_ADJACENT_TERMS)

def count_dictionary_hits(text: str, patterns: List[Tuple[str, re.Pattern]]) -> Dict[str, int]:
    counts = {}
    for term, pat in patterns:
        counts[term] = len(pat.findall(text))
    return counts

def sum_counts(counts: Dict[str, int]) -> int:
    return int(sum(counts.values()))


Metadata extraction

In [ ]:
DATE_LINE_RE = re.compile(
    r"^\s*([A-Z]+)\s+\d{1,2},\s+\d{4}\s*/\s*([0-9:APMapm\s]+)\s*GMT\s*$"
)
# Example line in your file: "JULY 28, 2022 / 9:00PM GMT" :contentReference[oaicite:2]{index=2}

def extract_call_date(text: str) -> Optional[str]:
    """
    Try to extract date in ISO (YYYY-MM-DD) from the transcript header.
    Returns None if not found.
    """
    month_map = {
        "JANUARY": "01", "FEBRUARY": "02", "MARCH": "03", "APRIL": "04",
        "MAY": "05", "JUNE": "06", "JULY": "07", "AUGUST": "08",
        "SEPTEMBER": "09", "OCTOBER": "10", "NOVEMBER": "11", "DECEMBER": "12"
    }

    for line in text.splitlines()[:80]:  # header is near the top
        line = line.strip()
        # Look for e.g. "JULY 28, 2022 / 9:00PM GMT"
        m = re.match(r"^([A-Z]+)\s+(\d{1,2}),\s+(\d{4})\s*/", line)
        if m:
            month, day, year = m.group(1), m.group(2), m.group(3)
            month = month.upper()
            if month in month_map:
                return f"{year}-{month_map[month]}-{int(day):02d}"
    return None

FISCAL_Q_RE = re.compile(r"\bQ([1-4])\s*(20\d{2})\b", flags=re.IGNORECASE)

def extract_fiscal_quarter_year(text: str) -> Tuple[Optional[int], Optional[int], Optional[str]]:
    """
    Extracts fiscal quarter and fiscal year from tokens like 'Q1 2022'.
    Returns (fiscal_q, fiscal_year, fiscal_qy_str) where fiscal_qy_str is e.g. 'Q1 2022'.
    """
    # Search near top where headers are
    head = "\n".join(text.splitlines()[:120])
    m = FISCAL_Q_RE.search(head)
    if not m:
        return None, None, None

    q = int(m.group(1))
    y = int(m.group(2))
    return q, y, f"Q{q} {y}"

Main runner: loop folders, export


In [ ]:
@dataclass
class TranscriptResult:
    company: str
    date: Optional[str]
    file: str
    total_words: int
    pres_words: int
    qa_words: int
    core_hits_total: int
    core_hits_pres: int
    core_hits_qa: int
    adj_hits_total: int
    adj_hits_pres: int
    adj_hits_qa: int
    fiscal_quarter: Optional[int]
    fiscal_year: Optional[int]
    fiscal_period: Optional[str]

def process_transcript(path: Path, company: str) -> TranscriptResult:
    raw = path.read_text(encoding="utf-8", errors="ignore")

    fiscal_quarter, fiscal_year, fiscal_period = extract_fiscal_quarter_year(raw)

    pres, qa = split_presentation_and_qa(raw)

    total_words = word_count(raw)
    pres_words = word_count(pres) if pres else 0
    qa_words = word_count(qa) if qa else 0

    core_total = sum_counts(count_dictionary_hits(raw, CORE_PATTERNS))
    core_pres = sum_counts(count_dictionary_hits(pres, CORE_PATTERNS)) if pres else 0
    core_qa = sum_counts(count_dictionary_hits(qa, CORE_PATTERNS)) if qa else 0

    adj_total = sum_counts(count_dictionary_hits(raw, ADJ_PATTERNS))
    adj_pres = sum_counts(count_dictionary_hits(pres, ADJ_PATTERNS)) if pres else 0
    adj_qa = sum_counts(count_dictionary_hits(qa, ADJ_PATTERNS)) if qa else 0

    call_date = extract_call_date(raw)

    return TranscriptResult(
        company=company,
        date=call_date,
        fiscal_quarter=fiscal_quarter,
        fiscal_year=fiscal_year,
        fiscal_period=fiscal_period,
        file=path.name,
        total_words=total_words,
        pres_words=pres_words,
        qa_words=qa_words,
        core_hits_total=core_total,
        core_hits_pres=core_pres,
        core_hits_qa=core_qa,
        adj_hits_total=adj_total,
        adj_hits_pres=adj_pres,
        adj_hits_qa=adj_qa,
    )

def run_corpus(input_root: str, out_csv: str, out_xlsx: str) -> pd.DataFrame:
    input_root = Path(input_root)
    rows = []

    # Each company has its own folder
    for company_dir in sorted([p for p in input_root.iterdir() if p.is_dir()]):
        company = company_dir.name
        for txt in sorted(company_dir.glob("*.txt")):
            r = process_transcript(txt, company)
            rows.append(r.__dict__)

    df = pd.DataFrame(rows)

    # Optional: add intensity measures (mentions per 1,000 words)
    def per_1000(n, denom):
        return (n / denom * 1000.0) if denom and denom > 0 else 0.0

    df["core_per_1000_total"] = [per_1000(n, w) for n, w in zip(df["core_hits_total"], df["total_words"])]
    df["core_per_1000_pres"] = [per_1000(n, w) for n, w in zip(df["core_hits_pres"], df["pres_words"])]
    df["core_per_1000_qa"] = [per_1000(n, w) for n, w in zip(df["core_hits_qa"], df["qa_words"])]

    df["adj_per_1000_total"] = [per_1000(n, w) for n, w in zip(df["adj_hits_total"], df["total_words"])]
    df["adj_per_1000_pres"] = [per_1000(n, w) for n, w in zip(df["adj_hits_pres"], df["pres_words"])]
    df["adj_per_1000_qa"] = [per_1000(n, w) for n, w in zip(df["adj_hits_qa"], df["qa_words"])]

    # Save
    df.to_csv(out_csv, index=False)
    df.to_excel(out_xlsx, index=False)

    return df


if __name__ == "__main__":
    # Example usage:
    # run_corpus("transcripts", "ai_counts.csv", "ai_counts.xlsx")
    pass

Execute

In [ ]:
df = run_corpus(
    input_root="Transcript",
    out_csv="ai_counts.csv",
    out_xlsx="ai_counts.xlsx"
)
df.head()